# Module 7.2: Automated Drift Detection with Kolmogorov-Smirnov Tests
### 🇵🇱 Automatyczna Detekcja Dryfu Testem Kołmogorowa-Smirnowa

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/07_02_ks_drift_detection.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Run statistical hypothesis tests comparing baseline reference distributions against live production streaming telemetry.  
> 🇵🇱 **Streszczenie:** Wykonuj testy statystyczne porównujące rozkład referencyjny z bieżącą telemetrią produkcyjną.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Data Drift vs Concept Drift**: Seasonal factory temperature increases shift sensor inputs P(X); tool wear alters physical failure relationships P(Y|X).
2. **Kolmogorov-Smirnov Testing**: Non-parametric two-sample test comparing cumulative distributions to flag calibration drift.
3. **Edge Optimization**: Converting models from FP32 to INT8 precision slashes inference latency by ~3.8x with < 0.5% accuracy drop.


In [ ]:
# 1. Simulate Production Telemetry Shift & Kolmogorov-Smirnov Test
from scipy import stats

np.random.seed(42)
# Baseline golden calibration data (e.g. ambient 22°C baseline)
baseline_telemetry = np.random.normal(loc=50.0, scale=4.0, size=1000)

# Deployed live telemetry during July summer heatwave (shifted by +1.5 standard deviations)
production_telemetry = np.random.normal(loc=56.2, scale=4.3, size=1000)

# Run two-sample Kolmogorov-Smirnov hypothesis test
ks_stat, p_value = stats.ks_2samp(baseline_telemetry, production_telemetry)
print(f"KS Statistic: {ks_stat:.4f}, p-value: {p_value:.2e}")


In [ ]:
# 2. Visualize Telemetry Drift Distribution Shift
plt.figure(figsize=(11, 5))
sns.kdeplot(baseline_telemetry, color='#0284c7', fill=True, alpha=0.3, label='Golden Baseline Reference', linewidth=2)
sns.kdeplot(production_telemetry, color='#ef4444', fill=True, alpha=0.3, label='Live Production Telemetry (Drifted)', linewidth=2)
plt.axvline(np.mean(baseline_telemetry), color='#0284c7', linestyle='--')
plt.axvline(np.mean(production_telemetry), color='#ef4444', linestyle='--')

plt.title(f"Production Telemetry Distribution Drift (KS p-value = {p_value:.2e})", fontsize=13, fontweight='bold')
plt.xlabel("Sensor Physical Parameter Value")
plt.ylabel("Probability Density")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Extract reference baseline distribution from verified training set *(🇵🇱 Wyodrębnij rozkład referencyjny z poprawnego zbioru treningowego)*
- [ ] **Task 2:** Run scipy.stats.ks_2samp across incoming daily telemetry windows *(🇵🇱 Uruchamiaj scipy.stats.ks_2samp dla kolejnych dziennych okien danych)*
- [ ] **Task 3:** Trigger automated alert when KS p-value drops below 0.01 *(🇵🇱 Wywołaj powiadomienie alarmowe, gdy wartość p-value testu KS spadnie poniżej 0.01)*


In [ ]:
# Exercise Challenge:
# Trigger automated MLOps retraining alarm if p-value < 0.01
is_drift_detected = p_value < 0.01
print(f"MLOps Monitor Status: {'[CRITICAL ALARM] Automated Retraining Triggered' if is_drift_detected else '[NORMAL] Model Calibrated'}")
assert is_drift_detected, "Drift detection failed to trigger on shifted telemetry!"
print("✓ MLOps monitoring exercise passed.")
